In [5]:
import pandas as pd

TASK2_40K_PATH = "/content/drive/MyDrive/CORTEX/T2/task2_weak_modality_for_task4_named.csv"   # <-- change this to your file path
t2_40k = pd.read_csv(TASK2_40K_PATH)

print("Loaded shape:", t2_40k.shape)
print("First 40 columns:", t2_40k.columns.tolist()[:40])

# Detect probability columns (common patterns)
prob_cols = [c for c in t2_40k.columns if c.startswith("task2_p_")]
if len(prob_cols) == 0:
    # fallback: try common names
    candidates = ["p_normal","p_anxiety","p_depression","p_suicidal",
                  "Normal","Anxiety","Depression","Suicidal"]
    prob_cols = [c for c in candidates if c in t2_40k.columns]

print("Detected prob cols:", prob_cols)

# Quick check
if len(prob_cols) > 0:
    print("\nHead of prob cols:")
    print(t2_40k[prob_cols].head(3))
    print("\nRow-sum min/max (should be ~1):",
          t2_40k[prob_cols].sum(axis=1).min(),
          t2_40k[prob_cols].sum(axis=1).max())
else:
    print("\n❌ No probability columns detected. We'll fix naming once you show column names.")

Loaded shape: (40012, 8)
First 40 columns: ['Unique_ID', 'task2_pred', 'task2_maxprob', 'task2_entropy', 'task2_p_normal', 'task2_p_anxiety', 'task2_p_depression', 'task2_p_suicidal']
Detected prob cols: ['task2_p_normal', 'task2_p_anxiety', 'task2_p_depression', 'task2_p_suicidal']

Head of prob cols:
   task2_p_normal  task2_p_anxiety  task2_p_depression  task2_p_suicidal
0        0.969466         0.022684            0.004817          0.003033
1        0.013613         0.979821            0.004435          0.002131
2        0.002566         0.993525            0.002890          0.001020

Row-sum min/max (should be ~1): 0.9999999999999993 1.0000000000000004


In [6]:
import numpy as np
import pandas as pd

# Reproducible
SEED = 42
rng = np.random.default_rng(SEED)

# --- Use your 4 probability columns ---
pN = t2_40k["task2_p_normal"].to_numpy()
pA = t2_40k["task2_p_anxiety"].to_numpy()
pD = t2_40k["task2_p_depression"].to_numpy()
pS = t2_40k["task2_p_suicidal"].to_numpy()

# =========================================================
# 1) Define tier-prior mapping from Task 2 probabilities
#    Tiers: 0 Supportive/Low, 1 Indicator, 2 Ideation, 3 Behavior, 4 Attempt
#
# This mapping is deliberately:
# - monotonic w.r.t suicidal probability
# - not deterministic (still allows overlap)
# =========================================================
w0 = 2.5*pN + 0.6*(1 - pS)                      # calm / low
w1 = 1.2*pA + 0.8*pD + 0.3*(1 - pS)             # mild/moderate indicators
w2 = 1.8*pD + 1.4*pA + 0.8*pS                   # ideation-like
w3 = 1.2*pS + 0.5*pD                            # behavior-like
w4 = 3.0*pS                                     # attempt-like (strongly tied to suicidal prob)

W = np.vstack([w0, w1, w2, w3, w4]).T
W = np.clip(W, 1e-9, None)
tier_probs = W / W.sum(axis=1, keepdims=True)

# sample tiers
tiers = np.array([rng.choice(5, p=tp) for tp in tier_probs], dtype=int)

print("Sampled tier distribution (counts):")
print(pd.Series(tiers).value_counts().sort_index())
print("Tier probs row-sum min/max:", tier_probs.sum(axis=1).min(), tier_probs.sum(axis=1).max())

# =========================================================
# 2) Reuse the same principled simulator (conditioned on tier)
# =========================================================

# ---- distributions (tunable, consistent with earlier setup) ----
p_I_present = {0: 0.05, 1: 0.35, 2: 0.85, 3: 0.95, 4: 0.98}

sev_dist = {
    0: {0: 1.0},
    1: {0:0.35, 1:0.40, 2:0.20, 3:0.05},
    2: {2:0.30, 3:0.40, 4:0.30},
    3: {3:0.20, 4:0.50, 5:0.30},
    4: {4:0.40, 5:0.60},
}

beta_params = {
    0: (1.2, 5.0),
    1: (1.5, 4.2),
    2: (2.6, 2.6),
    3: (3.5, 2.0),
    4: (4.5, 1.6),
}

p_B_preparatory = {0:0.01, 1:0.05, 2:0.00, 3:0.25, 4:0.35}
p_B_aborted     = {0:0.00, 1:0.02, 2:0.00, 3:0.20, 4:0.25}
p_B_interrupted = {0:0.00, 1:0.02, 2:0.00, 3:0.22, 4:0.30}
p_B_attempt     = {0:0.00, 1:0.01, 2:0.00, 3:0.10, 4:0.70}

EVENT_BUCKETS = [(0,7), (8,30), (31,3650)]
event_mix = {
    0: [0.00, 0.00, 1.00],
    1: [0.10, 0.25, 0.65],
    2: [0.30, 0.40, 0.30],
    3: [0.45, 0.40, 0.15],
    4: [0.60, 0.30, 0.10],
}

ONSET_BUCKETS = [(0,7), (8,30), (31,90), (91,3650)]
onset_mix = {
    0: [1.00, 0.00, 0.00, 0.00],
    1: [0.55, 0.30, 0.12, 0.03],
    2: [0.20, 0.40, 0.25, 0.15],
    3: [0.20, 0.35, 0.30, 0.15],
    4: [0.40, 0.30, 0.20, 0.10],
}

def sample_categorical(dist_dict):
    levels = np.array(list(dist_dict.keys()))
    probs  = np.array(list(dist_dict.values()), dtype=float)
    probs  = probs / probs.sum()
    return int(rng.choice(levels, p=probs))

def sample_intensity(tier):
    a, b = beta_params[tier]
    x = rng.beta(a, b)
    s = 1 + int(np.rint(4 * x))
    return int(np.clip(s, 1, 5))

def sample_from_buckets(buckets, weights):
    weights = np.array(weights, dtype=float)
    weights = weights / weights.sum()
    idx = int(rng.choice(len(buckets), p=weights))
    lo, hi = buckets[idx]
    return int(rng.integers(lo, hi+1))

def temporal_points(days_since_last_event, duration_since_onset, has_event: bool):
    # Only apply temporal points if ideation or behavior exists
    if not has_event:
        return 0, 0, 0

    if 0 <= days_since_last_event <= 7:
        acute = 8
    elif 8 <= days_since_last_event <= 30:
        acute = 4
    else:
        acute = 0

    if duration_since_onset <= 7:
        persist = 0
    elif duration_since_onset <= 30:
        persist = 2
    elif duration_since_onset <= 90:
        persist = 4
    else:
        persist = 7

    return acute, persist, acute + persist

def behavior_points(B_preparatory, B_aborted, B_interrupted, B_actual_attempt):
    pts = 0
    if B_preparatory:    pts = max(pts, 8)
    if B_aborted:        pts = max(pts, 12)
    if B_interrupted:    pts = max(pts, 18)
    if B_actual_attempt: pts = max(pts, 30)
    return float(pts)

def intensity_points(I_present, I_freq, I_duration, I_controllability, I_deterrents, I_reasons):
    if I_present == 0:
        return 0.0
    sub = np.array([I_freq, I_duration, I_controllability, I_deterrents, I_reasons], dtype=float)
    norm = (sub - 1.0) / 4.0
    return float(20.0 * norm.mean())

def ideation_points(I_severity_level):
    return float(7.0 * I_severity_level)  # 0..35

def risk_band(score, actual_attempt):
    if actual_attempt == 1 or score >= 85:
        return "Critical"
    elif score >= 65:
        return "High"
    elif score >= 35:
        return "Medium"
    else:
        return "Low"

# --- generate 40k structured rows ---
n = len(t2_40k)

age_group = np.where(rng.random(n) < 0.15, "Teen", "Adult")

I_present = np.zeros(n, dtype=int)
I_sev     = np.zeros(n, dtype=int)
I_freq = np.zeros(n, dtype=int)
I_dur  = np.zeros(n, dtype=int)
I_ctrl = np.zeros(n, dtype=int)
I_det  = np.zeros(n, dtype=int)
I_reas = np.zeros(n, dtype=int)

B_prep = np.zeros(n, dtype=int)
B_abort= np.zeros(n, dtype=int)
B_intr = np.zeros(n, dtype=int)
B_att  = np.zeros(n, dtype=int)
B_any  = np.zeros(n, dtype=int)

days_since_last_event = np.zeros(n, dtype=int)
duration_since_onset  = np.zeros(n, dtype=int)

for i, t in enumerate(tiers):
    # ideation presence
    I_present[i] = int(rng.random() < p_I_present[t])

    # ideation severity + intensity
    if I_present[i] == 1:
        I_sev[i] = sample_categorical(sev_dist[t])
        I_freq[i] = sample_intensity(t)
        I_dur[i]  = sample_intensity(t)
        I_ctrl[i] = sample_intensity(t)
        I_det[i]  = sample_intensity(t)
        I_reas[i] = sample_intensity(t)
    else:
        I_sev[i] = 0

    # behavior sampling
    B_prep[i]  = int(rng.random() < p_B_preparatory[t])
    B_abort[i] = int(rng.random() < p_B_aborted[t])
    B_intr[i]  = int(rng.random() < p_B_interrupted[t])
    B_att[i]   = int(rng.random() < p_B_attempt[t])

    # HARD tier constraints
    if t == 4:
        B_att[i] = 1
        I_present[i] = 1
    elif t == 3:
        if (B_prep[i] + B_abort[i] + B_intr[i] + B_att[i]) == 0:
            choice = rng.choice(["prep","abort","intr"], p=[0.4,0.3,0.3])
            if choice == "prep":  B_prep[i] = 1
            elif choice == "abort": B_abort[i] = 1
            else: B_intr[i] = 1
        I_present[i] = 1
    elif t == 2:
        # ideation tier: no behavior
        I_present[i] = 1
        B_prep[i] = B_abort[i] = B_intr[i] = B_att[i] = 0

    B_any[i] = int((B_prep[i] or B_abort[i] or B_intr[i] or B_att[i]))

    # if attempt, enforce higher ideation
    if B_att[i] == 1:
        if I_sev[i] < 4:
            I_sev[i] = sample_categorical({4:0.4, 5:0.6})
        if I_freq[i] == 0:
            tt = max(int(t), 3)
            I_freq[i] = sample_intensity(tt)
            I_dur[i]  = sample_intensity(tt)
            I_ctrl[i] = sample_intensity(tt)
            I_det[i]  = sample_intensity(tt)
            I_reas[i] = sample_intensity(tt)

    # temporal features
    has_event = (I_present[i] == 1) or (B_any[i] == 1)
    if has_event:
        days_since_last_event[i] = sample_from_buckets(EVENT_BUCKETS, event_mix[t])
        duration_since_onset[i]  = sample_from_buckets(ONSET_BUCKETS, onset_mix[t])
        if duration_since_onset[i] < days_since_last_event[i]:
            duration_since_onset[i] = min(days_since_last_event[i] + int(rng.integers(0, 30)), 3650)

# compute points + severity
pts_behavior = np.array([behavior_points(bp,ba,bi,bt) for bp,ba,bi,bt in zip(B_prep,B_abort,B_intr,B_att)])
pts_ideation = 7.0 * I_sev

pts_intensity = np.array([
    intensity_points(ip, f, d, c, de, r)
    for ip, f, d, c, de, r in zip(I_present, I_freq, I_dur, I_ctrl, I_det, I_reas)
])

pts_acute = np.zeros(n, dtype=float)
pts_persist = np.zeros(n, dtype=float)
pts_temporal = np.zeros(n, dtype=float)

for i in range(n):
    has_event = (I_present[i] == 1) or (B_any[i] == 1)
    a, p, tp = temporal_points(days_since_last_event[i], duration_since_onset[i], has_event)
    pts_acute[i] = a
    pts_persist[i] = p
    pts_temporal[i] = tp

severity = np.clip(pts_behavior + pts_ideation + pts_intensity + pts_temporal, 0, 100)
bands = [risk_band(s, a) for s, a in zip(severity, B_att)]

task3_40k = pd.DataFrame({
    "Unique_ID": t2_40k["Unique_ID"].values,
    "age_group": age_group,
    "tier_sim": tiers,

    "I_present": I_present,
    "I_severity_level": I_sev,
    "I_freq": I_freq,
    "I_duration": I_dur,
    "I_controllability": I_ctrl,
    "I_deterrents": I_det,
    "I_reasons": I_reas,

    "B_preparatory": B_prep,
    "B_aborted": B_abort,
    "B_interrupted": B_intr,
    "B_actual_attempt": B_att,
    "B_any": B_any,

    "days_since_last_event": days_since_last_event,
    "duration_since_onset": duration_since_onset,

    "points_behavior": pts_behavior,
    "points_ideation": pts_ideation,
    "points_intensity": pts_intensity,
    "points_acute": pts_acute,
    "points_persistence": pts_persist,
    "points_temporal": pts_temporal,

    "severity_score_rule": severity,
    "risk_band": bands,
})

print("\nTask3_40k created:", task3_40k.shape)
print(task3_40k["risk_band"].value_counts())
print(task3_40k.groupby("tier_sim")["severity_score_rule"].mean().sort_index().round(3))

task3_40k.head(3)

Sampled tier distribution (counts):
0    15319
1     5917
2     8101
3     3947
4     6728
Name: count, dtype: int64
Tier probs row-sum min/max: 0.9999999999999997 1.0000000000000004

Task3_40k created: (40012, 25)
risk_band
Low         24142
Critical     7211
Medium       7173
High         1486
Name: count, dtype: int64
tier_sim
0     0.670
1     8.782
2    34.061
3    63.408
4    86.218
Name: severity_score_rule, dtype: float64


,Unique_ID,age_group,tier_sim,I_present,I_severity_level,I_freq,I_duration,I_controllability,I_deterrents,I_reasons,...,days_since_last_event,duration_since_onset,points_behavior,points_ideation,points_intensity,points_acute,points_persistence,points_temporal,severity_score_rule,risk_band
0,0,Adult,0,0,0,0,0,0,0,0,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Low
1,1,Teen,1,0,0,0,0,0,0,0,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Low
2,2,Adult,2,1,4,3,2,4,3,4,...,28,48,0.0,28.0,11.0,4.0,4.0,8.0,47.0,Medium


In [7]:
# Define save path (change folder if needed)
SAVE_PATH = "/content/drive/MyDrive/CORTEX/T3/task3_strong_modality_40k.csv"

# Save as CSV
task3_40k.to_csv(SAVE_PATH, index=False)

print("Saved successfully to:")
print(SAVE_PATH)

# Optional: also save compressed version (recommended)
SAVE_PATH_ZIP = "/content/drive/MyDrive/CORTEX/T3/task3_strong_modality_40k.csv.gz"
task3_40k.to_csv(SAVE_PATH_ZIP, index=False, compression="gzip")

print("\nCompressed version saved to:")
print(SAVE_PATH_ZIP)

Saved successfully to:
/content/drive/MyDrive/CORTEX/T3/task3_strong_modality_40k.csv

Compressed version saved to:
/content/drive/MyDrive/CORTEX/T3/task3_strong_modality_40k.csv.gz
